# LIBRARIES

In [ ]:
from __future__ import annotations
import csv
import os
import subprocess
import sqlite3
# Standard Library
from datetime import datetime

# Prompts, Hub, and Messages
from langchain import hub
from langchain_core.messages import AIMessage, SystemMessage
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
)

# LLMs and Chat Models
from langchain_openai import ChatOpenAI, OpenAI

# Core Chains (Legacy fallback paths)
from langchain_classic.chains import LLMChain, SequentialChain, TransformChain
from langchain_classic.chains.llm_math.base import LLMMathChain

# Tools and Utilities
from langchain_core.tools import Tool
from langchain_community.utilities import SQLDatabase
from langchain_experimental.tools import PythonREPLTool

# Agents and Agent Toolkits
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent, create_openai_tools_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.agent_toolkits.sql.prompt import SQL_FUNCTIONS_SUFFIX

from src.constants import  OPENAI_API_KEY, LLM_MODEL_NAME, DATABASE

# SETUP SQL CONNECTION

In [ ]:
mydb = sqlite3.connect(DATABASE)
mycursor = mydb.cursor()

# CREATE ALL TABLES

In [ ]:
mycursor.execute("CREATE TABLE IF NOT EXISTS distribution_centers(id INT, name VARCHAR(255), latitude FLOAT, longitude FLOAT)")
mycursor.execute("CREATE TABLE IF NOT EXISTS events(id INT, user_id INT, sequence_number INT, session_id VARCHAR(255), created_at TIMESTAMP, ip_address VARCHAR(255), city VARCHAR(255), state VARCHAR(255), postal_code VARCHAR(255), browser VARCHAR(255), traffic_source VARCHAR(255), uri VARCHAR(255), event_type VARCHAR(255))")
mycursor.execute("CREATE TABLE IF NOT EXISTS inventory_items(id INT, product_id INT, created_at TIMESTAMP, sold_at TIMESTAMP, cost FLOAT, product_category VARCHAR(255), product_name VARCHAR(255), product_brand VARCHAR(255), product_retail_price FLOAT, product_department VARCHAR(255), product_sku VARCHAR(255), product_distribution_center_id INT)")
mycursor.execute("CREATE TABLE IF NOT EXISTS order_items(id INT, order_id INT, user_id INT, product_id INT, inventory_item_id INT, status VARCHAR(255), created_at TIMESTAMP, shipped_at TIMESTAMP, delivered_at TIMESTAMP, returned_at TIMESTAMP, sale_price FLOAT)")
mycursor.execute("CREATE TABLE IF NOT EXISTS orders(order_id INT, user_id INT, status VARCHAR(255), gender VARCHAR(255), created_at TIMESTAMP, returned_at TIMESTAMP, shipped_at TIMESTAMP, delivered_at TIMESTAMP, num_of_item INT)")
mycursor.execute("CREATE TABLE IF NOT EXISTS products(id INT, cost FLOAT, category VARCHAR(255), name VARCHAR(255), brand VARCHAR(255), retail_price FLOAT, department VARCHAR(255), sku VARCHAR(255), distribution_center_id INT)")
mycursor.execute("CREATE TABLE IF NOT EXISTS users(id INT, first_name VARCHAR(255), last_name VARCHAR(255), email VARCHAR(255), age INT, gender VARCHAR(255), state VARCHAR(255), street_address VARCHAR(255), postal_code VARCHAR(255), city VARCHAR(255), country VARCHAR(255), latitude FLOAT, longitude FLOAT, traffic_source VARCHAR(255), created_at TIMESTAMP)")
mydb.commit()
# Close connection when done
mydb.close()


# DATA EXPORT TO MYSQL TABLES

In [ ]:
mydb = sqlite3.connect(DATABASE)
mycursor = mydb.cursor()

table_names = ["distribution_centers", "events", "inventory_items", "order_items", "orders", "products", "users"]

# Function to detect timestamp columns based on data types
def detect_timestamp_columns(cursor, table_name):
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()
    timestamp_indices = [i for i, column in enumerate(columns) if column[2].startswith('TIMESTAMP')]
    return timestamp_indices

for table_name in table_names:
    timestamp_indices = detect_timestamp_columns(mycursor, table_name)

    with open(f"data/{table_name}.csv", 'r', encoding='utf-8') as csv_file:  # Specifying utf-8 encoding
        csv_data = csv.reader(csv_file)
        next(csv_data)  # Skip headers
        counter = 0
        print(f"Currently inserting data into table {table_name}")

        for row in csv_data:
            if counter % 10000 == 0:
                print(f"Progress is {counter}")

            row = [None if cell == "" else cell for cell in row]

            # Correct datetime values if necessary
            for col_index in timestamp_indices:
                if row[col_index] is not None and row[col_index] != '':
                    try:
                        clean_val = row[col_index].replace(" UTC", "")
                        datetime.strptime(clean_val, "%Y-%m-%d %H:%M:%S")
                        row[col_index] = clean_val  # store cleaned string    
                    except ValueError:
                        # Handle invalid datetime value by setting it to None
                        row[col_index] = None
                else:
                    # Handle None or empty string
                    row[col_index] = None

            postfix = ','.join(["?"] * len(row))
            query = f"INSERT INTO {table_name} VALUES ({postfix})"
            try:
                mycursor.execute(query, row)
            except sqlite3.Error as err:
                print(f"Error: {err}")
                print(f"Failed row: {row}")

            counter += 1

        mydb.commit()
        print(f"Completed inserting data into table {table_name}")
mydb.close()

# SQL TABLE CONNECTION

In [ ]:
sql_db = SQLDatabase.from_uri(f"sqlite:///{DATABASE}")

In [ ]:
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
model_name = LLM_MODEL_NAME

In [ ]:
print(os.environ['OPENAI_API_KEY'])

In [ ]:
llm = OpenAI(temperature=0.0, openai_api_key=OPENAI_API_KEY)

# SQL QUERY ENGINE

- Creates SQL query from the user input

In [ ]:
from langchain_community.tools.sql_database.tool import (
    InfoSQLDatabaseTool,
    ListSQLDatabaseTool,
    QuerySQLCheckerTool,
    QuerySQLDataBaseTool,
)

class SQLQueryEngine:
    """
    A class representing an SQL query engine.

    Attributes:
        llm (ChatOpenAI): An instance of ChatOpenAI used for natural language processing.
        toolkit (SQLDatabaseToolkit): An SQL database toolkit instance.
        context (dict): Contextual information obtained from the SQL database toolkit.
        tools (list): List of tools available for SQL query execution.
        prompt (ChatPromptTemplate): The prompt used for interactions with the SQL query engine.
        agent_executor (AgentExecutor): An executor for the SQL query engine's agent.
    """

    def __init__(self, model_name, db):
        self.llm = ChatOpenAI(model=model_name, temperature=0)
        self.tools = [
            QuerySQLDataBaseTool(db=db),
            InfoSQLDatabaseTool(db=db),
            ListSQLDatabaseTool(db=db)
            # QuerySQLCheckerTool(db=db, llm=self.llm)
        ]
        
        self.toolkit = SQLDatabaseToolkit(db=db, llm=self.llm)
        self.context = self.toolkit.db.get_context()
        #self.tools = self.toolkit.get_tools()
        self.prompt = None
        self.agent_executor = None

    def set_prompt(self):
        messages = [
            HumanMessagePromptTemplate.from_template("{input}"),
            AIMessage(content=SQL_FUNCTIONS_SUFFIX),
            MessagesPlaceholder(variable_name="agent_scratchpad")
            ]
        self.prompt = ChatPromptTemplate.from_messages(messages)
        self.prompt = self.prompt.partial(**self.context)

    def initialize_agent(self):
        agent = create_openai_tools_agent(self.llm, self.tools, self.prompt)
        self.agent_executor = AgentExecutor(
            agent=agent,
            tools=self.tools,
            verbose=True,
        )

    def get_query_data(self, query):
        if 'return' in query:
            query = query + "\n" + "return percentage is defined as total number of returns divided by total number of orders. You can join orders table with users table to know more about each user"
        return self.agent_executor.invoke({"input": query})['output']

# PYTHON DASHBOARD ENGINE

- Creates Streamlit dashboard from data output by SQLQueryEngine

In [ ]:
# REPL -> Read Evaluate Print Loop
class PythonDashboardEngine:
    """
    A class representing a Python dashboard engine.

    Attributes:
        tools (list): A list of tools available for the dashboard engine.
        instructions (str): Instructions guiding the behavior of the dashboard engine.
        prompt (str): The prompt used for interactions with the dashboard engine.
        agent_executor (AgentExecutor): An executor for the dashboard engine's agent.
    """
    def __init__(self):
        self.tools = [PythonREPLTool()]
        self.instructions = """You are an agent designed to write a python code to answer questions.
        You have access to a python REPL, which you can use to execute python code.
        If you get an error, debug your code and try again.
        You might know the answer without running any code, but you should still run the code to get the answer.
        If it does not seem like you can write code to answer the question, just return "I don't know" as the answer.
        Always output the python code only.
        """
        base_prompt = ChatPromptTemplate.from_messages([
            ("system", "{instructions}"),
            MessagesPlaceholder(variable_name="chat_history", optional=True),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ])
        self.prompt = base_prompt.partial(instructions=self.instructions)
        self.agent_executor = None

    def initialize(self):
        agent = create_openai_functions_agent(ChatOpenAI(model=model_name, temperature=0), self.tools, self.prompt)
        self.agent_executor = AgentExecutor(agent=agent, tools=self.tools, verbose=True)

    def get_output(self, query):
        output = self.agent_executor.invoke({"input": "Write a code in python to plot the following data\n\n" + query})
        return output['output']

    def parse_output(self, inp):
        if "```" in inp:
            parts = inp.split("```")
            # Try to grab the code block (second element)
            code = parts[1]
        else:
            # Assume whole string is code
            code = inp

        # Clean up
        code = code.replace("python", "").replace("plt.show()", "")

        outp = (
        "import streamlit as st\n"
        "import matplotlib.pyplot as plt\n"
        "st.title('E-commerce Company Insights')\n"
        "st.write('Here is our LLM generated dashboard')\n"
        + code +
        "\nfig = plt.gcf()\n"
        "st.pyplot(fig)\n"
        )
        return outp


    def export_to_streamlit(self, data):
        with open("app.py", "w") as text_file:
            text_file.write(self.parse_output(data))

        command = "streamlit run app.py --global.developmentMode=false --server.headless=true"
        proc = subprocess.Popen([command], shell=True, stdin=None, stdout=None, stderr=None, close_fds=True)

# INITIATE SQL AND DASHBOARD ENGINES

In [ ]:
global sql_query_engine, dashboard_engine

def init_engines():
    sql_query_engine = SQLQueryEngine(model_name, sql_db)
    sql_query_engine.set_prompt()
    sql_query_engine.initialize_agent()

    dashboard_engine = PythonDashboardEngine()
    dashboard_engine.initialize()
    return sql_query_engine, dashboard_engine

In [ ]:
init_engines()

# EXAMPLE QUERIES

## QUERY 1

- Number of users with their gender

In [ ]:
sql_query_engine, dashboard_engine = init_engines()
query = "Number of users with their gender"
sql_query_engine_output = sql_query_engine.get_query_data(query)
#print(sql_query_engine_output)

dashboard_engine_output = dashboard_engine.get_output(sql_query_engine_output)
dashboard_engine.export_to_streamlit(dashboard_engine_output)

## QUERY 2

- number of users in each country who came via facebook

In [ ]:
#sql_query_engine, dashboard_engine = init_engines()
query = "number of users in each country who came via facebook"
sql_query_engine_output = sql_query_engine.get_query_data(query)
#print(sql_query_engine_output)

dashboard_engine_output = dashboard_engine.get_output(sql_query_engine_output)
dashboard_engine.export_to_streamlit(dashboard_engine_output)

## QUERY 3

- number of orders per month since January 2020

In [ ]:
#sql_query_engine, dashboard_engine = init_engines()
query = "number of orders per month since January 2020"
sql_query_engine_output = sql_query_engine.get_query_data(query)
#print(sql_query_engine_output)

dashboard_engine_output = dashboard_engine.get_output(sql_query_engine_output)
dashboard_engine.export_to_streamlit(dashboard_engine_output)

## QUERY 4

- top 3 product categories with highest number of returns by count

In [ ]:
#sql_query_engine, dashboard_engine = init_engines()
query = "top 3 product categories with highest number of returns by count"
sql_query_engine_output = sql_query_engine.get_query_data(query)
#print(sql_query_engine_output)

dashboard_engine_output = dashboard_engine.get_output(sql_query_engine_output)
dashboard_engine.export_to_streamlit(dashboard_engine_output)

## QUERY 5

- return percentage country wise

In [ ]:
#sql_query_engine, dashboard_engine = init_engines()
query = "return percentage country wise"
sql_query_engine_output = sql_query_engine.get_query_data(query)
#print(sql_query_engine_output)

dashboard_engine_output = dashboard_engine.get_output(sql_query_engine_output)
dashboard_engine.export_to_streamlit(dashboard_engine_output)

## QUERY 6

- which are my top 5 geographies with highest business

In [ ]:
#sql_query_engine, dashboard_engine = init_engines()
query = "which are my top 5 geographies with highest business"
sql_query_engine_output = sql_query_engine.get_query_data(query)
#print(sql_query_engine_output)

dashboard_engine_output = dashboard_engine.get_output(sql_query_engine_output)
dashboard_engine.export_to_streamlit(dashboard_engine_output)

## QUERY 7

- which product categories have the highest margins in 2024

In [ ]:
#sql_query_engine, dashboard_engine = init_engines()
query = "which product categories have the highest margins in 2024"
sql_query_engine_output = sql_query_engine.get_query_data(query)
#print(sql_query_engine_output)

dashboard_engine_output = dashboard_engine.get_output(sql_query_engine_output)
dashboard_engine.export_to_streamlit(dashboard_engine_output)